# 📧 Email Categorizer Engine (No Pre-Trained APIs)
**Hackathon Submission: Custom TF-IDF, Truncated SVD, and Latent Centroid Matching**

### 🏛️ Technical Architecture Write-Up
1. **Vectorization Technique (Custom TF-IDF):** Built from scratch without `sklearn`. Document frequency is computed across all corpus terms, term frequency is scaled via $\log(1 + \text{count})$, and IDF via $\log\left(\frac{N + 1}{df + 1}\right) + 1$. Each document vector is L2-normalized.
2. **Core Dimensionality Reduction (Truncated SVD):** Uses numpy matrix factorization $A = U \Sigma V^T$. Keeps top $k=50$ singular vectors ($V_k$) to project high-dimensional sparse TF-IDF vectors into a dense, 50-dimensional semantic latent space ($X_{svd} = A V_k$).
3. **Domain Fit & Topic Structure:** Email text contains noisy vocabulary, informal language, and overlapping jargon. SVD compresses synonymy and co-occurrence patterns into orthogonal latent factors, allowing centroid-based cosine similarity to reliably match emails to category clusters (Education, Work, Finance, Promotions).

In [ ]:
import json
import math
import os
import re
import numpy as np
import pandas as pd

# 1. Load Data
data_path = '../data/sample_emails.json'
if not os.path.exists(data_path):
    data_path = 'email_categorizer/data/sample_emails.json'

with open(data_path, 'r') as f:
    emails = json.load(f)

print(f'Successfully loaded {len(emails)} synthetic emails.')

In [ ]:
# 2. Custom TF-IDF Implementation (No sklearn)
class CustomTfidf:
    def __init__(self, stop_words=None):
        self.stop_words = stop_words or {
            'the', 'a', 'an', 'is', 'it', 'in', 'on', 'of', 'for', 'to', 'and', 
            'or', 'with', 'this', 'that', 'by', 'from', 'at', 'be', 'are', 'was', 
            'were', 'your', 'our', 'my', 'have', 'has', 'had', 'you', 'we', 'will', 
            'can', 'should', 'all', 'more', 'new', 'get', 'not', 'if', 'so', 'as', 
            'but', 'they', 'their', 'them', 'who', 'which', 'what', 'when', 'where'
        }
        self.vocabulary = {}
        self.feature_names = []
        self.idf = []
        self.N = 0

    def _tokenize(self, text):
        text = text.lower()
        tokens = re.findall(r'\b[a-z0-9]{2,}\b', text)
        return [t for t in tokens if t not in self.stop_words]

    def fit_transform(self, documents):
        self.N = len(documents)
        tokenized_docs = [self._tokenize(doc) for doc in documents]

        vocab_set = set()
        for doc in tokenized_docs:
            vocab_set.update(doc)
        
        self.feature_names = sorted(list(vocab_set))
        self.vocabulary = {word: idx for idx, word in enumerate(self.feature_names)}
        V = len(self.feature_names)

        df = [0] * V
        for doc in tokenized_docs:
            unique_words = set(doc)
            for w in unique_words:
                df[self.vocabulary[w]] += 1

        self.idf = np.array([math.log((self.N + 1) / (df_val + 1)) + 1.0 for df_val in df])

        A = np.zeros((self.N, V), dtype=np.float64)
        for doc_idx, doc in enumerate(tokenized_docs):
            word_counts = {}
            for w in doc:
                word_counts[w] = word_counts.get(w, 0) + 1

            for w, count in word_counts.items():
                if w in self.vocabulary:
                    col_idx = self.vocabulary[w]
                    tf = math.log(1 + count)
                    A[doc_idx, col_idx] = tf * self.idf[col_idx]

            norm = np.linalg.norm(A[doc_idx])
            if norm > 0:
                A[doc_idx] /= norm

        return A

    def transform(self, documents):
        V = len(self.feature_names)
        tokenized_docs = [self._tokenize(doc) for doc in documents]
        A = np.zeros((len(documents), V), dtype=np.float64)

        for doc_idx, doc in enumerate(tokenized_docs):
            word_counts = {}
            for w in doc:
                word_counts[w] = word_counts.get(w, 0) + 1

            for w, count in word_counts.items():
                if w in self.vocabulary:
                    col_idx = self.vocabulary[w]
                    tf = math.log(1 + count)
                    A[doc_idx, col_idx] = tf * self.idf[col_idx]

            norm = np.linalg.norm(A[doc_idx])
            if norm > 0:
                A[doc_idx] /= norm

        return A

# Fit TF-IDF matrix
corpus = [f"{e['subject']} {e['body']}" for e in emails]
tfidf = CustomTfidf()
A = tfidf.fit_transform(corpus)
print(f'TF-IDF Dense Matrix created with shape: {A.shape}')

In [ ]:
# 3. Truncated SVD (50 Latent Components)
U, S, Vt = np.linalg.svd(A, full_matrices=False)
k = min(50, A.shape[0], A.shape[1])
V_k = Vt[:k, :].T  # Singular vectors (V x k)

# Project document vectors into k-dimensional latent space
X_svd = A @ V_k  # (N x k)

# L2 normalize latent representations
norms = np.linalg.norm(X_svd, axis=1, keepdims=True)
norms[norms == 0] = 1.0
X_svd = X_svd / norms

print(f'SVD Latent Matrix shape: {X_svd.shape} (k={k} components)')

In [ ]:
# 4. Seed Keywords & Category Centroid Generation
seed_dict = {
    'Education': ['syllabus', 'lecture', 'assignment', 'campus', 'course', 'homework', 'exam', 'university', 'professor', 'student', 'grade', 'class', 'academic', 'study', 'library', 'lab', 'thesis'],
    'Work': ['invoice', 'meeting', 'client', 'q4', 'project', 'deadline', 'report', 'presentation', 'team', 'agenda', 'schedule', 'manager', 'sprint', 'contract', 'review', 'okr', 'deliverables'],
    'Finance': ['transaction', 'balance', 'mortgage', 'stock', 'bank', 'payment', 'credit', 'account', 'transfer', 'interest', 'statement', 'tax', 'loan', 'debit', 'dividend', 'escrow', 'wire'],
    'Promotions': ['sale', 'discount', 'coupon', 'newsletter', 'deal', 'offer', 'shop', 'free', 'clearance', 'promo', 'buy', 'limited', 'vip', 'reward', 'voucher', 'flash', 'apparel']
}

centroids = {}
for cat, seeds in seed_dict.items():
    seed_text = ' '.join(seeds)
    seed_vec = tfidf.transform([seed_text])
    svd_centroid = seed_vec @ V_k
    c_norm = np.linalg.norm(svd_centroid)
    if c_norm > 0:
        svd_centroid /= c_norm
    centroids[cat] = svd_centroid[0].tolist()

print('Generated 50D Centroids for 4 Categories:')
for cat in centroids:
    print(f' - {cat}: 50D Vector (L2 norm = {np.linalg.norm(centroids[cat]):.4f})')

In [ ]:
# 5. Cosine Similarity & Kaggle Validation Table
cat_names = list(centroids.keys())
centroid_matrix = np.array([centroids[cat] for cat in cat_names])
sim_matrix = X_svd @ centroid_matrix.T

predictions = []
confidences = []
correct = 0

for i, email in enumerate(emails):
    scores = sim_matrix[i]
    best_idx = np.argmax(scores)
    best_cat = cat_names[best_idx]
    conf = float(scores[best_idx])
    conf_bounded = max(0.0, min(1.0, conf))
    
    email['assigned_category'] = best_cat
    email['confidence'] = round(conf_bounded, 4)
    email['svd_vector'] = X_svd[i].tolist()
    
    predictions.append(best_cat)
    confidences.append(f'{round(conf_bounded * 100, 1)}%')
    
    if email.get('true_category') == best_cat:
        correct += 1

accuracy = (correct / len(emails)) * 100
print(f'Overall Accuracy on Synthetic Corpus: {accuracy:.2f}% ({correct}/{len(emails)})
')

# Validation Table DataFrame
df_val = pd.DataFrame({
    'Email Subject': [e['subject'] for e in emails[:20]],
    'True Category': [e.get('true_category', 'N/A') for e in emails[:20]],
    'Assigned Category': predictions[:20],
    'Confidence': confidences[:20]
})
display(df_val)

In [ ]:
# 6. Export Centroids & Metadata for Chrome Extension
assets_dir = '../extension/assets'
if not os.path.exists(assets_dir):
    assets_dir = 'email_categorizer/extension/assets'
os.makedirs(assets_dir, exist_ok=True)

np.save(os.path.join(assets_dir, 'centroids.npy'), np.array([centroids[cat] for cat in cat_names]))
with open(os.path.join(assets_dir, 'centroids.json'), 'w') as f:
    json.dump(centroids, f, indent=2)

with open(os.path.join(assets_dir, 'email_metadata.json'), 'w') as f:
    json.dump(emails, f, indent=2)

print(f'Exported centroids.npy, centroids.json, and email_metadata.json to {assets_dir}/')